# 06 · E8 Run time and scalability (ULB)

Two parts:

1. **Run time per configuration**, taken from E1 (no training here).
2. **Scalability**: the full pipeline on stratified subsamples from 50,000 rows to the whole dataset, one run per size, seed 42.

Timings depend on the Colab machine. `run_info.json` records the device for every run. Make sure the run time you report in the paper comes from the same runtime type as the rest. If E1 and E8 are run on different machines, the numbers will not line up, and that is the likely cause of the mismatch in the v5 results.

| | |
|---|---|
| Trains models | yes (part 2 only): one full pipeline per size |
| Expected time | about 5 min on a GPU runtime |
| Writes | `results/E8_cost/`: `cost_by_configuration.csv`, `scalability_single_run.csv` |

In [ ]:
import os, sys

ROOT = os.environ.get("RAREGUARD_ROOT", "/content/drive/MyDrive/PhD/RAREGuard_v6")

if ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.path.join(ROOT, "code"))
import rareguard_lib as rg

try:
    from IPython.display import display
except ImportError:
    display = print

cfg = rg.load_config(ROOT)
env = rg.show_environment()

In [ ]:
import time, gc
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

EXP = "E8_cost"
FORCE_RERUN = False
RUN_TRAINING = FORCE_RERUN or not rg.already_done(cfg, EXP, ["scalability_single_run.csv"])
run = rg.Run(cfg, EXP, "06_E8_cost_scalability.ipynb", "Run time per configuration and scalability")

e1 = rg.read_result(cfg, "E1_components", "runs_by_seed.csv")
e1 = e1[e1.Threshold_Mode == "Fully unsupervised"]
cost = rg.mean_sd(e1, ["Experiment"], ["Train_Time_s", "Total_Time_s"])
display(cost.round(2))
run.save_csv(cost, "cost_by_configuration.csv")

In [ ]:
if RUN_TRAINING:
    X, y, feature_names = rg.load_ulb(cfg)
    W, C, seed = cfg["fusion_weights"], cfg["contamination"], cfg["e8"]["seed"]
    rows = []
    for size in cfg["e8"]["sizes"]:
        n = len(y) if size == "all" else int(size)
        if n > len(y):
            continue
        rg.set_seed(seed)
        t0 = time.time()
        sub = np.arange(len(y)) if n == len(y) else train_test_split(
            np.arange(len(y)), train_size=n, stratify=y, random_state=seed)[0]
        Xs, ys = X[sub], y[sub]
        tr, va, te = rg.make_split(ys, seed, cfg)
        F = rg.prepare_features(Xs[tr], Xs[va], Xs[te], feature_names, seed, cfg)
        scores, secs, _ = rg.fit_detectors(F, ys[tr], seed, cfg)
        val_s = rg.weighted_fusion({m: scores[m][1] for m in scores}, W)
        test_s = rg.weighted_fusion({m: scores[m][2] for m in scores}, W)
        m = rg.evaluate(ys[te], test_s, rg.percentile_threshold(val_s, C), val_s)
        rows.append({"Sample_Size": n, "Seed": seed, "Prep_s": F["seconds"],
                     **{f"{k}_s": v for k, v in secs.items()},
                     "Train_Time_s": sum(secs.values()), "Total_Time_s": time.time() - t0,
                     "Model_Dims": F["dims"], **m})
        print(f"{n:>7,} rows | total {rows[-1]['Total_Time_s']:6.1f}s | PR-AUC {m['PR_AUC']:.3f}")
        gc.collect()
    run.save_csv(pd.DataFrame(rows), "scalability_single_run.csv")
run.finish(notes="single run per size" if RUN_TRAINING else "cost table only")